# 04 — Backtest Tearsheet

Run a full backtest, generate a tearsheet with equity curve, drawdown,
and monthly returns.  Claude interprets the results and suggests next steps.

**Cost per full run**: ~£0.04 (two Claude API calls at ~1k tokens each).

Runs without an API key — all charts render; interpretations show unavailable.

---

### Real Data Configuration

| Parameter | Value |
|---|---|
| **Universe** | 10 FTSE 100 tickers: SHEL.L, AZN.L, HSBA.L, ULVR.L, BP.L, RIO.L, GSK.L, LLOY.L, VOD.L, BT-A.L |
| **Date range** | 2018-01-01 to 2024-12-31 |
| **Training period** | 2018-01-01 to 2023-12-31 (1,513 trading days) |
| **Out-of-sample test** | 2024-01-01 to 2024-12-30 (253 trading days) |
| **Walk-forward split** | Strict temporal — 2024 data never touches feature tuning or model training |
| **Strategy** | RSI mean reversion |
| **Portfolio optimisation** | Mean-variance (riskfolio-lib) on training returns |
| **Transaction costs** | 10 bps commission + 5 bps slippage |

The multi-ticker portfolio backtest is run via `scripts/run_real_backtest.py`.
Section 6 below loads and visualises the saved results.

### Path setup

`sys.path.insert(0, "..")` makes `src` importable from `notebooks/`.

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.config import load_config
from src.services.data_service import DataService
from src.features.pipeline import FeaturePipeline
from src.backtest.engine import BacktestEngine
from src.backtest.strategy import MeanReversionStrategy
from src.notebooks.formatters import (
    format_backtest_metrics,
    format_risk_metrics,
    display_interpretation,
)
from src.notebooks.research_log import ResearchLog

In [ ]:
config = load_config()
data_service = DataService(config)
pipeline = FeaturePipeline(config)
engine = BacktestEngine(config)
research_log = ResearchLog(config)

ticker = config["universe"]["tickers"][0]
print(f"Backtest tearsheet for: {ticker}")

## Section 1 — Run Full Backtest

In [ ]:
ohlcv = data_service.get_price_data(ticker)
features = pipeline.generate(ohlcv)

strategy = MeanReversionStrategy("rsi_mean_reversion")
result = engine.run(strategy, ohlcv, features)

print(f"Strategy: {result.strategy_name}")
print(f"Period: {result.equity_curve.index.min()} -> {result.equity_curve.index.max()}")
print(f"Initial capital: {config['backtest']['initial_capital']:,.0f} {config['backtest']['currency']}")

## Section 2 — Equity Curve, Drawdown, Monthly Returns

In [ ]:
# Equity curve.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result.equity_curve.index,
    y=result.equity_curve.values,
    mode="lines",
    name="Portfolio Value",
    line=dict(color="#1a73e8"),
))
fig.update_layout(
    title="Equity Curve",
    xaxis_title="Date",
    yaxis_title=f"Value ({config['backtest']['currency']})",
    yaxis_tickformat=",.0f",
)
fig.show()

In [ ]:
# Drawdown chart.
cumulative = (1 + result.returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=drawdown.index,
    y=drawdown.values * 100,
    fill="tozeroy",
    mode="lines",
    name="Drawdown",
    line=dict(color="#d32f2f"),
    fillcolor="rgba(211, 47, 47, 0.3)",
))
fig.update_layout(
    title="Drawdown",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    yaxis_ticksuffix="%",
)
fig.show()

In [ ]:
# Monthly returns heatmap.
monthly_returns = result.returns.resample("ME").apply(
    lambda x: (1 + x).prod() - 1
)
monthly_df = pd.DataFrame({
    "year": monthly_returns.index.year,
    "month": monthly_returns.index.month,
    "return": (monthly_returns.values * 100).round(2),
})

if len(monthly_df) > 0:
    pivot = monthly_df.pivot(index="year", columns="month", values="return")
    pivot.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:len(pivot.columns)]

    fig = px.imshow(
        pivot,
        text_auto=".1f",
        title="Monthly Returns (%)",
        color_continuous_scale="RdYlGn",
        zmin=-5,
        zmax=5,
    )
    fig.show()
else:
    print("Insufficient data for monthly returns heatmap.")

## Section 3 — Metrics Table

In [ ]:
metrics = result.metrics
formatted_metrics = format_backtest_metrics(metrics)

metrics_display = pd.DataFrame(
    list(formatted_metrics.items()),
    columns=["Metric", "Value"],
)
metrics_display

## Section 4 — Claude Tearsheet Interpretation

In [ ]:
tearsheet_data = formatted_metrics.copy()
tearsheet_data["ticker"] = ticker
tearsheet_data["strategy"] = result.strategy_name
tearsheet_data["period"] = f"{result.equity_curve.index.min()} -> {result.equity_curve.index.max()}"
tearsheet_data["initial_capital"] = config["backtest"]["initial_capital"]
tearsheet_data["final_value"] = round(float(result.equity_curve.iloc[-1]), 2)

try:
    from src.notebooks.claude_interpreter import QuantInterpreter

    interpreter = QuantInterpreter(config)
    tearsheet_interpretation = interpreter.interpret("backtest_tearsheet", tearsheet_data)
except (EnvironmentError, ImportError) as e:
    print(f"Claude interpretation unavailable: {e}")
    tearsheet_interpretation = {
        "summary": "Interpretation unavailable (no API key set)",
        "observations": [],
        "warnings": ["Set ANTHROPIC_API_KEY to enable interpretations"],
        "suggestions": [],
        "confidence": "low",
    }

display_interpretation(tearsheet_interpretation)

## Section 5 — Claude Next Steps

In [ ]:
# Feed the tearsheet back to Claude for next-steps suggestions.
next_steps_data = {
    "tearsheet": tearsheet_data,
    "tearsheet_interpretation": tearsheet_interpretation,
}

try:
    next_steps = interpreter.interpret("next_steps", next_steps_data)
except (EnvironmentError, NameError) as e:
    print(f"Claude interpretation unavailable: {e}")
    next_steps = {
        "summary": "Interpretation unavailable (no API key set)",
        "observations": [],
        "warnings": ["Set ANTHROPIC_API_KEY to enable interpretations"],
        "suggestions": [],
        "confidence": "low",
    }

display_interpretation(next_steps)

In [ ]:
# Log both interpretations.
research_log.log_entry(
    notebook="04_backtest_tearsheet",
    task="backtest_tearsheet",
    data_summary=tearsheet_data,
    interpretation=tearsheet_interpretation,
)
research_log.log_entry(
    notebook="04_backtest_tearsheet",
    task="next_steps",
    data_summary=next_steps_data,
    interpretation=next_steps,
)
print("Both entries logged.")

# Session summary.
session = research_log.summarise_session("04_backtest_tearsheet")
print(f"Session entries: {session['count']}")

## Section 6 — Real Data Multi-Ticker Portfolio Results

Loads the pre-computed results from `scripts/run_real_backtest.py`.

In [ ]:
import json
from pathlib import Path

data_dir = Path(config["general"]["data_dir"])
tearsheet_path = data_dir / "processed" / "real_tearsheet.json"
equity_path = data_dir / "processed" / "real_equity_curve.csv"
interp_path = data_dir / "processed" / "real_interpretation.json"

if tearsheet_path.exists():
    with open(tearsheet_path) as f:
        real_tearsheet = json.load(f)

    # Display key metrics
    print("=" * 60)
    print("  REAL DATA OOS RESULTS (10 FTSE 100 tickers)")
    print("=" * 60)
    print(f"  Period:           {real_tearsheet['oos_start']} -> {real_tearsheet['oos_end']}")
    print(f"  Tickers:          {real_tearsheet['n_tickers']}")
    print(f"  Strategy:         {real_tearsheet['strategy']}")
    print(f"  Optimisation:     {real_tearsheet['optimisation_method']}")
    print(f"  Initial Capital:  {real_tearsheet['initial_capital']:,.0f} GBP")
    print(f"  Final Value:      {real_tearsheet['final_value']:,.2f} GBP")
    print(f"  Total Return:     {real_tearsheet['total_return_pct']:.2f}%")
    print(f"  Sharpe Ratio:     {real_tearsheet['sharpe']:.2f}")
    print(f"  Max Drawdown:     {real_tearsheet['max_drawdown']:.4f}")
    print("=" * 60)

    # Weights table
    weights_df = pd.DataFrame(
        list(real_tearsheet["weights"].items()),
        columns=["Ticker", "Weight"],
    )
    weights_df["Weight"] = (weights_df["Weight"] * 100).round(2).astype(str) + "%"
    display(weights_df)
else:
    print("No real tearsheet found. Run: python -m scripts.run_real_backtest")

In [ ]:
# Real data equity curve.
if equity_path.exists():
    equity_df = pd.read_csv(equity_path, index_col=0, parse_dates=True)
    equity_series = equity_df.iloc[:, 0] if isinstance(equity_df, pd.DataFrame) else equity_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=equity_series.index,
        y=equity_series.values,
        mode="lines",
        name="Portfolio Value",
        line=dict(color="#1a73e8"),
    ))
    fig.update_layout(
        title="Real Data OOS Equity Curve (10 FTSE 100 Tickers, 2024)",
        xaxis_title="Date",
        yaxis_title="Value (GBP)",
        yaxis_tickformat=",.0f",
    )
    fig.show()

    # Drawdown
    cumulative = equity_series / equity_series.iloc[0]
    running_max = cumulative.cummax()
    dd = (cumulative - running_max) / running_max

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=dd.index,
        y=dd.values * 100,
        fill="tozeroy",
        mode="lines",
        name="Drawdown",
        line=dict(color="#d32f2f"),
        fillcolor="rgba(211, 47, 47, 0.3)",
    ))
    fig.update_layout(
        title="Real Data OOS Drawdown (2024)",
        xaxis_title="Date",
        yaxis_title="Drawdown (%)",
        yaxis_ticksuffix="%",
    )
    fig.show()
else:
    print("No equity curve found. Run: python -m scripts.run_real_backtest")

In [ ]:
# Real data interpretation.
if interp_path.exists():
    with open(interp_path) as f:
        real_interpretation = json.load(f)
    display_interpretation(real_interpretation)
else:
    print("No interpretation found. Run: python -m scripts.run_real_backtest")